# Multimodal types: images, audio, video, and attachments

This notebook covers Chapter 7 section 7.3. DSPy extends beyond text with `dspy.Image`, `dspy.Audio`, custom types you can build yourself (we'll build a `Video` type), and the `attachments` library for documents.

**Environment variables**: `OPENAI_API_KEY` for image classification; a Gemini key is required if you actually run the video example (the `Video.format()` method enforces a Gemini model). `Audio` requires a provider that supports audio input — Gemini works well across all multimodal inputs.

## Asset placeholders

Some cells reference local media files. The audio and video cells use placeholder filenames — drop in your own `meeting_recording.wav` and `sample_video.mp4` (or any small audio/video file) in `chapter07/assets/` before running them. No binary fixtures ship with this notebook.


In [ ]:
%pip install -r ../requirements.txt -q
%pip install attachments soundfile -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## 7.3.1 Images

`dspy.Image` accepts URLs, local file paths, PIL images, bytes, and data URIs. The old factory methods `Image.from_url()` and `Image.from_file()` still work but are deprecated, so prefer the constructor directly. This runnable version creates a small in-memory product image instead of depending on placeholder URLs or local files.


In [ ]:
from PIL import Image as PILImage, ImageDraw

# The constructor also accepts URLs and local file paths. A generated PIL image
# keeps this notebook self-contained while exercising the same DSPy type.
pil_image = PILImage.new("RGB", (320, 200), "white")
draw = ImageDraw.Draw(pil_image)
draw.arc((70, 20, 250, 180), 180, 360, fill="black", width=12)
draw.rounded_rectangle((55, 95, 105, 180), radius=12, fill="royalblue")
draw.rounded_rectangle((215, 95, 265, 180), radius=12, fill="royalblue")

img = dspy.Image(pil_image)

classify = dspy.Predict(
    "image: dspy.Image, category_options: list[str] -> category: str, confidence: float"
)

result = classify(
    image=img,
    category_options=["electronics", "clothing", "furniture", "food"]
)
print(f"{result.category} ({result.confidence})")


## 7.3.2 Audio

`dspy.Audio` handles URLs, local files, and raw numpy arrays. Not all providers support audio input. The manuscript shows file and URL constructors; this runnable notebook generates a short waveform so the type can be exercised without missing media assets.


In [ ]:
import dspy
import numpy as np

# The manuscript also demonstrates:
# dspy.Audio.from_file("meeting_recording.wav")
# dspy.Audio.from_url("https://example.com/clip.mp3")

# Generate one second of a quiet 440 Hz tone. `soundfile` is installed above.
sampling_rate = 16_000
t = np.arange(sampling_rate) / sampling_rate
audio_array = (0.05 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
audio = dspy.Audio.from_array(audio_array, sampling_rate=sampling_rate)

extract_actions = dspy.ChainOfThought(
    "recording: dspy.Audio, context: str -> action_items: list[str]"
)

print(audio)
print("To run transcription, pass `audio` to `extract_actions` with a model that supports audio input.")


## 7.3.3 Subclassing Type for videos

No built-in Video type exists in DSPy, but the type system is designed to be extended. Below is a complete custom `Video` type that wraps a video into a format Gemini (via LiteLLM) understands. The key trick is to label the content block as `image_url`, not `video_url`.


In [ ]:
import base64
import mimetypes
import os
from typing import Any, Union
from urllib.parse import urlparse

import requests

from dspy.adapters.types.base_type import Type


class Video(Type):
    url: str

    model_config = {
        "frozen": True,
        "str_strip_whitespace": True,
        "validate_assignment": True,
        "extra": "forbid",
    }
    
    def __init__(self, url_or_dict=None, **kwargs):
        """Initialize Video with flexible input handling."""
        if url_or_dict is None and "url" in kwargs:
            # Called with Video(url="...")
            super().__init__(**kwargs)
        elif isinstance(url_or_dict, str):
            # Called with Video("...")
            super().__init__(url=url_or_dict)
        elif isinstance(url_or_dict, dict):
            # Called with Video({"url": "..."})
            super().__init__(**url_or_dict)
        elif url_or_dict is None:
            # Called with Video()
            super().__init__(**kwargs)
        else:
            raise TypeError("Expected a string URL or a dictionary with a key 'url'.")

    def format(self) -> list[dict[str, Any]] | str:
        # Check if the model is a Gemini model
        import dspy
        current_lm = dspy.settings.lm
        if current_lm and hasattr(current_lm, 'model'):
            model_name = current_lm.model.lower()
            if 'gemini' not in model_name and 'vertex' not in model_name:
                raise ValueError(
                    f"Video input is currently only supported on Gemini models. "
                    f"Current model: {current_lm.model}. "
                    f"Please use a Gemini model (e.g., 'gemini/gemini-2.0-flash' or similar) to use video inputs."
                )
        
        try:
            # For Gemini, URLs are supported directly (unlike the previous assumption)
            # Only download and encode for local files
            if self.url.startswith("http"):
                video_url = encode_video(self.url, download_videos=False)
            else:
                video_url = encode_video(self.url)
        except Exception as e:
            raise ValueError(f"Failed to format video for DSPy: {e}")
        
        # Gemini expects videos to use the image_url format (based on testing with litellm)
        # Using "video_url" type causes Gemini to say it can't access videos
        return [{"type": "image_url", "image_url": {"url": video_url}}]


    @classmethod
    def from_url(cls, url: str, download: bool = False):
        # For remote URLs, we don't need to download since Gemini supports URLs directly
        # For local files, this will encode them as base64
        return cls(url=encode_video(url, download_videos=download))

    @classmethod
    def from_file(cls, file_path: str):
        # Convert to absolute path if relative
        abs_path = os.path.abspath(file_path)
        return cls(url=encode_video(abs_path))

    def __str__(self):
        return self.serialize_model()

    def __repr__(self):
        if "base64" in self.url:
            len_base64 = len(self.url.split("base64,")[1])
            video_type = self.url.split(";")[0].split("/")[-1]
            return f"Video(url=data:video/{video_type};base64,<VIDEO_BASE_64_ENCODED({len_base64!s})>)"
        return f"Video(url='{self.url}')"


def is_url(string: str) -> bool:
    """Check if a string is a valid URL."""
    try:
        result = urlparse(string)
        return all([result.scheme in ("http", "https", "gs"), result.netloc])
    except ValueError:
        return False


def encode_video(video: Union[str, bytes, dict], download_videos: bool = False) -> str:
    """
    Encode a video to a base64 data URI.

    Args:
        video: The video to encode. Can be a file path, URL, or data URI.
        download_videos: Whether to download videos from URLs.

    Returns:
        str: The data URI of the video or the URL if download_videos is False.

    Raises:
        ValueError: If the video type is not supported.
    """
    if isinstance(video, dict) and "url" in video:
        # NOTE: Not doing other validation for now
        return video["url"]
    elif isinstance(video, str):
        if video.startswith("data:"):
            # Already a data URI
            return video
        elif os.path.isfile(video):
            # File path
            return _encode_video_from_file(video)
        elif is_url(video):
            # URL
            if download_videos:
                return _encode_video_from_url(video)
            else:
                # Return the URL as is (Gemini supports URLs directly)
                return video
        else:
            # Unsupported string format
            print(f"Unsupported video string: {video}")
            raise ValueError(f"Unsupported video string: {video}")
    elif isinstance(video, bytes):
        # Raw bytes
        raise ValueError("Direct byte encoding for videos is not supported. Please provide a file path or URL.")
    elif isinstance(video, Video):
        return video.url
    else:
        print(f"Unsupported video type: {type(video)}")
        raise ValueError(f"Unsupported video type: {type(video)}")


def _encode_video_from_file(file_path: str) -> str:
    """Encode a video from a file path to a base64 data URI."""
    # Check if file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Video file not found: {file_path}")
    
    # Check if it's actually a file
    if not os.path.isfile(file_path):
        raise ValueError(f"Path is not a file: {file_path}")
    
    # Check file size - Gemini has limits
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    if file_size_mb > 2000:  # 2GB limit
        raise ValueError(f"Video file size ({file_size_mb:.1f}MB) exceeds Gemini's 2GB limit")
    
    with open(file_path, "rb") as file:
        file_data = file.read()

    # Use mimetypes to guess directly from the file path
    mime_type, _ = mimetypes.guess_type(file_path)
    if mime_type is None:
        # Default to mp4 if we can't determine the type
        mime_type = "video/mp4"
    
    # Verify it's a video mime type
    if not mime_type.startswith("video/"):
        raise ValueError(f"File does not appear to be a video. MIME type: {mime_type}")

    encoded_data = base64.b64encode(file_data).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_data}"


def _encode_video_from_url(video_url: str) -> str:
    """Encode a video from a URL to a base64 data URI."""
    response = requests.get(video_url, stream=True)
    response.raise_for_status()
    
    # Check content length if available
    content_length = response.headers.get('Content-Length')
    if content_length:
        size_mb = int(content_length) / (1024 * 1024)
        if size_mb > 2000:  # 2GB limit
            raise ValueError(f"Video file size ({size_mb:.1f}MB) exceeds Gemini's 2GB limit")
    
    content_type = response.headers.get("Content-Type", "")

    # Use the content type from the response headers if available
    if content_type:
        mime_type = content_type
    else:
        # Try to guess MIME type from URL
        mime_type, _ = mimetypes.guess_type(video_url)
        if mime_type is None:
            mime_type = "video/mp4"
    
    # Verify it's a video mime type
    if not mime_type.startswith("video/"):
        raise ValueError(f"URL does not appear to point to a video. MIME type: {mime_type}")

    # Read the content
    content_chunks = []
    total_size = 0
    for chunk in response.iter_content(chunk_size=8192):
        content_chunks.append(chunk)
        total_size += len(chunk)
        if total_size > 2000 * 1024 * 1024:  # 2GB limit
            raise ValueError(f"Video file size exceeds Gemini's 2GB limit")
    
    content = b''.join(content_chunks)
    encoded_data = base64.b64encode(content).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_data}"


def is_video(obj) -> bool:
    """Check if the object is a video or a valid video file reference."""
    if isinstance(obj, str):
        if obj.startswith("data:video"):
            return True
        elif os.path.isfile(obj):
            mime_type, _ = mimetypes.guess_type(obj)
            return mime_type is not None and mime_type.startswith("video/")
        elif is_url(obj):
            # For URLs, we can't easily check without downloading
            # So we'll be permissive and assume URLs ending in video extensions are videos
            ext = os.path.splitext(urlparse(obj).path)[1].lower()
            return ext in ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.wmv', '.m4v', '.mpg', '.mpeg']
    return False


Use the `Video` type the same way as any other DSPy multimodal type. Place your own `sample_video.mp4` in `chapter07/assets/` before running the example below.


In [ ]:
# place your own sample_video.mp4 in chapter07/assets/ here
# Requires a Gemini model in dspy.configure(...)
#
# import dspy
# dspy.configure(lm=dspy.LM("gemini/gemini-2.0-flash"))
#
# analyze = dspy.Predict("clip: Video -> summary: str")
# result = analyze(clip=Video.from_file("assets/sample_video.mp4"))
# print(result.summary)


## 7.3.4 Documents with the Attachments library

[Attachments](https://github.com/maximerivest/attachments) is a community library that funnels files — PDFs, Word docs, PowerPoint, CSVs, URLs, even zip files and git repos — into a format LLMs can process. Its DSPy integration creates a `dspy.Type` subclass you can use directly in Signatures.


In [ ]:
from attachments.dspy import Attachments

# Attachments handles file conversion automatically. The manuscript uses
# `invoice.pdf`; this repository includes an equivalent text fixture so the
# notebook runs without a binary download.
extract = dspy.Predict(
    "document: Attachments -> vendor: str, total: float, line_items: list[str]"
)

result = extract(
    document=Attachments("assets/invoice.md")
)
print(f"Vendor: {result.vendor}, Total: ${result.total}")
